In [6]:
%cd /drive2/ryusejong/LFF
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
import json 
import time 
import re
import random
import types
import math
import numpy as np 
from tqdm.auto import tqdm
from util.utils import set_seed, read_data, save_result, get_answer_from_text, chat_huggingface, chat_huggingface_with_hidden_states, construct_conversation, normalize_answer, get_answer_response_from_text
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel

seed = 42
set_seed(seed)

/drive2/ryusejong/LFF


/drive2/ryusejong/miniconda3/envs/llm1/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
### laod prm model
prm_path = "UW-Madison-Lee-Lab/Llama-PRM800K"
device = "cuda:0" if torch.cuda.is_available() else "cpu"

candidate_tokens = [12, 10]
prm_tokenizer = AutoTokenizer.from_pretrained(prm_path)
prm_tokenizer.pad_token = prm_tokenizer.eos_token
prm_tokenizer.padding_side = 'left' 
prm_tokenizer.truncation_side = 'left'
    
prm = AutoModelForCausalLM.from_pretrained(
    prm_path,
    torch_dtype=torch.bfloat16,
    device_map=device
)
prm.eval()

In [8]:
def basic_prm_score(question, reasoning, prm, prm_tokenzier):
    print("-"*50)
    
    reasoning_steps = [l.strip() for l in reasoning.split("\n") if l.strip()]
    print(f"# steps: {len(reasoning_steps)}")

    prm_input_text = question + ' \n\n' + ' \n\n\n\n'.join(reasoning_steps) + ' \n\n\n\n'

    ### original score
    with torch.no_grad():
        prm_input = torch.tensor([prm_tokenizer.encode(prm_input_text)]).to(prm.device)
        prm_logits = prm(prm_input).logits[:,:,candidate_tokens]
        #print(logits.shape)
        prm_scores = prm_logits.softmax(dim=-1)[:,:,1]
        #print(scores.shape)
        step_scores = prm_scores[prm_input == 23535]
        step_probs  = step_scores.tolist()
    
    print(f"Step scores: {step_scores}")

In [9]:
def cur_step_prm_score(question, reasoning, prm, prm_tokenizer):
    print("-"*50)
    
    reasoning_steps = [l.strip() for l in reasoning.split("\n") if l.strip()]
    print(f"# steps: {len(reasoning_steps)}")
    
    for i, step in enumerate(reasoning_steps):
        prm_input_text = question + ' \n\n' + step + ' \n\n\n\n'
        
        with torch.no_grad():
            prm_input = torch.tensor([prm_tokenizer.encode(prm_input_text)]).to(prm.device)
            prm_logits = prm(prm_input).logits[:,:,candidate_tokens]
            #print(logits.shape)
            prm_scores = prm_logits.softmax(dim=-1)[:,:,1]
            #print(scores.shape)
            step_scores = prm_scores[prm_input == 23535]
            step_probs  = step_scores.tolist()
        #print(question)
        print(f"Step: {step} \nScore: {step_probs}")

In [7]:
def pre_cur_step_prm_score(question, reasoning, prm, prm_tokenizer):
    print("-"*50)
    
    reasoning_steps = [l.strip() for l in reasoning.split("\n") if l.strip()]
    print(f"# steps: {len(reasoning_steps)}")
    
    for i, step in enumerate(reasoning_steps):
        if i == 0:
            prm_input_text = question + ' \n\n' + step + ' \n\n\n\n'
        else:
            prm_input_text = question + ' \n\n' + reasoning_steps[i-1] + ' \n\n\n\n' + step + ' \n\n\n\n'
        with torch.no_grad():
            prm_input = torch.tensor([prm_tokenizer.encode(prm_input_text)]).to(prm.device)
            prm_logits = prm(prm_input).logits[:,:,candidate_tokens]
            #print(logits.shape)
            prm_scores = prm_logits.softmax(dim=-1)[:,:,1]
            #print(scores.shape)
            step_scores = prm_scores[prm_input == 23535]
            step_probs  = step_scores.tolist()
        #print(question)
        print(f"Step: {step} \nScore: {step_probs}")

## 1 Previous step + current step

In [12]:
### incorrect case
question = "Bryan starts exercising at home during quarantine. To start, he decides to do 3 sets of 15 push-ups each. Near the end of the third set, he gets tired and does 5 fewer push-ups. How many push-ups did he do in total? Explain your reasoning step-by-step. Your final answer should be put between two ##, like ## 1 ## (if your final answer is 1), at the end of your response."

reasoning = "Let's break it down step-by-step!\n\n1. Bryan starts with 3 sets of 15 push-ups each. So, he does 3 x 15 = 45 push-ups in the first two sets.\n2. In the third set, he does 5 fewer push-ups than usual. So, he does 15 - 5 = 10 push-ups in the third set.\n3. To find the total number of push-ups, we add the number of push-ups in the first two sets (45) to the number of push-ups in the third set (10).\n\n45 + 10 = 55\n\n## 55 ##"

### basic
basic_prm_score(question, reasoning, prm, prm_tokenizer)

### pre_cur
pre_cur_step_prm_score(question, reasoning, prm, prm_tokenizer)


--------------------------------------------------
# steps: 6
Step scores: tensor([0.9492, 0.6094, 0.9648, 0.9727, 0.9414, 0.9414], device='cuda:0',
       dtype=torch.bfloat16)
--------------------------------------------------
# steps: 6
Step: Let's break it down step-by-step! 
Score: [0.94921875]
Step: 1. Bryan starts with 3 sets of 15 push-ups each. So, he does 3 x 15 = 45 push-ups in the first two sets. 
Score: [0.94921875, 0.609375]
Step: 2. In the third set, he does 5 fewer push-ups than usual. So, he does 15 - 5 = 10 push-ups in the third set. 
Score: [0.546875, 0.96484375]
Step: 3. To find the total number of push-ups, we add the number of push-ups in the first two sets (45) to the number of push-ups in the third set (10). 
Score: [0.7734375, 0.6484375]
Step: 45 + 10 = 55 
Score: [0.20703125, 0.5625]
Step: ## 55 ## 
Score: [0.11865234375, 0.4296875]


In [13]:
### incorrect case
question = "Dylan's mother is throwing a baby shower for her best friend. She is expecting 40 guests, of whom she has cleared the parking lot to park in to, leaving only her car and her husband's jeep in the parking lot. The 40 guests, though, arrive in only 10 cars that they park in the parking lot.  If each car has 4 wheels, how many car wheels are there in the parking lot, including both of Dylan's parent's car wheels? Explain your reasoning step-by-step. Your final answer should be put between two ##, like ## 1 ## (if your final answer is 1), at the end of your response."

reasoning = "Let's break this down step by step!\n\n1. There are 40 guests arriving in 10 cars. This means that each car has 40/10 = 4 guests.\n2. Each car has 4 wheels, so each car has a total of 4 x 4 = 16 wheels.\n3. There are 10 cars in total, so the total number of wheels is 10 x 16 = 160 wheels.\n4. Dylan's mother and father have 2 cars in the parking lot, which are their own car and their husband's jeep. Each of these cars has 4 wheels, so the total number of wheels for these 2 cars is 2 x 4 = 8 wheels.\n5. Adding the wheels from the 10 guest cars (160) to the wheels from Dylan's parents' cars (8), we get a total of 160 + 8 = 168 wheels in the parking lot.\n\n## 168 ##"

### basic
basic_prm_score(question, reasoning, prm, prm_tokenizer)

### pre_cur
pre_cur_step_prm_score(question, reasoning, prm, prm_tokenizer)

--------------------------------------------------
# steps: 7
Step scores: tensor([0.9219, 0.8789, 0.9102, 0.7930, 0.6523, 0.9336, 0.8945],
       device='cuda:0', dtype=torch.bfloat16)
--------------------------------------------------
# steps: 7
Step: Let's break this down step by step! 
Score: [0.921875]
Step: 1. There are 40 guests arriving in 10 cars. This means that each car has 40/10 = 4 guests. 
Score: [0.921875, 0.87890625]
Step: 2. Each car has 4 wheels, so each car has a total of 4 x 4 = 16 wheels. 
Score: [0.81640625, 0.9140625]
Step: 3. There are 10 cars in total, so the total number of wheels is 10 x 16 = 160 wheels. 
Score: [0.34765625, 0.62109375]
Step: 4. Dylan's mother and father have 2 cars in the parking lot, which are their own car and their husband's jeep. Each of these cars has 4 wheels, so the total number of wheels for these 2 cars is 2 x 4 = 8 wheels. 
Score: [0.07861328125, 0.67578125]
Step: 5. Adding the wheels from the 10 guest cars (160) to the wheels from

In [14]:
### incorrect case
question = "Donna is trying to figure out if her truck is over a particular bridge's weight limit. The bridge can hold 20,000 pounds. Donna's empty truck weighs 12,000 pounds. She's carrying 20 crates of soda that each weigh 50 pounds, 3 dryers that each weigh 3000 pounds, and twice as much weight in fresh produce as in soda. How much does Donna's fully loaded truck weigh? Explain your reasoning step-by-step. Your final answer should be put between two ##, like ## 1 ## (if your final answer is 1), at the end of your response."

reasoning = "Let's break this down step by step!\n\n1. Donna's empty truck weighs 12,000 pounds.\n2. Each crate of soda weighs 50 pounds, and there are 20 crates. So, the total weight of the soda is:\n20 crates x 50 pounds/crate = 1000 pounds\n3. The 3 dryers each weigh 3000 pounds, so the total weight of the dryers is:\n3 dryers x 3000 pounds/dryer = 9000 pounds\n4. The problem states that Donna is carrying twice as much weight in fresh produce as in soda. Since the soda weighs 1000 pounds, the fresh produce weighs:\n1000 pounds x 2 = 2000 pounds\n5. Now, let's add up the weights of the truck, soda, dryers, and fresh produce:\n12,000 pounds (empty truck) + 1000 pounds (soda) + 9000 pounds (dryers) + 2000 pounds (fresh produce) = 22,000 pounds\n\nSo, Donna's fully loaded truck weighs 22,000 pounds. ## 22,000 ##"

### basic
basic_prm_score(question, reasoning, prm, prm_tokenizer)

### pre_cur
pre_cur_step_prm_score(question, reasoning, prm, prm_tokenizer)

--------------------------------------------------
# steps: 11
Step scores: tensor([0.9531, 0.9922, 0.8555, 0.9766, 0.9688, 0.9883, 0.9688, 0.9688, 0.9570,
        0.7969, 0.7500], device='cuda:0', dtype=torch.bfloat16)
--------------------------------------------------
# steps: 11
Step: Let's break this down step by step! 
Score: [0.953125]
Step: 1. Donna's empty truck weighs 12,000 pounds. 
Score: [0.953125, 0.9921875]
Step: 2. Each crate of soda weighs 50 pounds, and there are 20 crates. So, the total weight of the soda is: 
Score: [0.9609375, 0.8671875]
Step: 20 crates x 50 pounds/crate = 1000 pounds 
Score: [0.671875, 0.96484375]
Step: 3. The 3 dryers each weigh 3000 pounds, so the total weight of the dryers is: 
Score: [0.8359375, 0.83203125]
Step: 3 dryers x 3000 pounds/dryer = 9000 pounds 
Score: [0.609375, 0.9453125]
Step: 4. The problem states that Donna is carrying twice as much weight in fresh produce as in soda. Since the soda weighs 1000 pounds, the fresh produce weighs: 

In [16]:
### correct case
question = "Jane picked 64 apples at the orchard. She sorted them into 4 different baskets to send to her friends. When Jane wasn't looking her sister took 3 apples from each of the baskets. How many apples are in each basket now? Explain your reasoning step-by-step. Your final answer should be put between two ##, like ## 1 ## (if your final answer is 1), at the end of your response."

reasoning = "Let's break this problem down step by step!\n\n1. Jane picked 64 apples and sorted them into 4 baskets. To find out how many apples are in each basket initially, we can divide the total number of apples (64) by the number of baskets (4):\n\n64 \u00f7 4 = 16\n\nSo, each basket initially had 16 apples.\n\n2. Jane's sister took 3 apples from each of the baskets. To find out how many apples are left in each basket, we need to subtract 3 from the initial number of apples in each basket (16):\n\n16 - 3 = 13\n\nSo, each basket now has 13 apples.\n\n## 13 ##"

### basic
basic_prm_score(question, reasoning, prm, prm_tokenizer)

### pre_cur
pre_cur_step_prm_score(question, reasoning, prm, prm_tokenizer)

--------------------------------------------------
# steps: 8
Step scores: tensor([0.9570, 0.9258, 0.9922, 0.9961, 0.9883, 0.9922, 0.9922, 0.9609],
       device='cuda:0', dtype=torch.bfloat16)
--------------------------------------------------
# steps: 8
Step: Let's break this problem down step by step! 
Score: [0.95703125]
Step: 1. Jane picked 64 apples and sorted them into 4 baskets. To find out how many apples are in each basket initially, we can divide the total number of apples (64) by the number of baskets (4): 
Score: [0.95703125, 0.92578125]
Step: 64 ÷ 4 = 16 
Score: [0.8359375, 0.984375]
Step: So, each basket initially had 16 apples. 
Score: [0.89453125, 0.9921875]
Step: 2. Jane's sister took 3 apples from each of the baskets. To find out how many apples are left in each basket, we need to subtract 3 from the initial number of apples in each basket (16): 
Score: [0.94140625, 0.86328125]
Step: 16 - 3 = 13 
Score: [0.396484375, 0.67578125]
Step: So, each basket now has 13 apple

## Prompt to consider problem and previous reasoning steps 

In [1]:
def prompt_cur_step_prm_score(question, reasoning, prm, prm_tokenizer):
    print("-"*50)
    
    reasoning_steps = [l.strip() for l in reasoning.split("\n") if l.strip()]
    print(f"# steps: {len(reasoning_steps)}")
    
    for i, step in enumerate(reasoning_steps):
        if i == 0:
            prm_input_text = question + ' \n\n' + f"Based on problem, next step is: {step}" + ' \n\n\n\n'
            #prm_input_text = question + ' \n\n' + f"Considering the conditions from the question, proceed with the next logical step. {step}" + ' \n\n\n\n'
        else:
            prm_input_text = question  + ' \n\n' + ' \n\n\n\n'.join(reasoning_steps[:i]) + ' \n\n\n\n' + f" Based on problem and previous reasoning, next step is: {step}" + ' \n\n\n\n'
            #prm_input_text = question  + ' \n\n' + ' \n\n\n\n'.join(reasoning_steps[:i]) + ' \n\n\n\n' + f" Considering the conditions from the question and the reasoning steps so far, proceed with the next logical step. {step}" + ' \n\n\n\n'
        with torch.no_grad():
            prm_input = torch.tensor([prm_tokenizer.encode(prm_input_text)]).to(prm.device)
            prm_logits = prm(prm_input).logits[:,:,candidate_tokens]
            #print(logits.shape)
            prm_scores = prm_logits.softmax(dim=-1)[:,:,1]
            #print(scores.shape)
            step_scores = prm_scores[prm_input == 23535]
            step_probs  = step_scores.tolist()
        #print(question)
        print(f"Step: {step} \nScore: {step_probs}")

In [174]:
def prompt_all_step_prm_score(question, reasoning, prm, prm_tokenizer):
    print("-"*50)
    
    reasoning_steps = [l.strip() for l in reasoning.split("\n") if l.strip()]
    #prefix = ["Based on problem, next step is: ", " Based on problem and previous reasoning, next step is: "]
    prefix = ["Considering the conditions from the question, proceed with the next logical step. ", " Considering the conditions from the question and the reasoning steps so far, proceed with the next logical step. "]
    reasoning_steps = [prefix[min(i,1)] + f"{step}" for i, step in enumerate(reasoning_steps)]

    print(f"# steps: {len(reasoning_steps)}")
    
    for i, step in enumerate(reasoning_steps):
        if i == 0:
            prm_input_text = question + ' \n\n' + step + ' \n\n\n\n'
        else:
            prm_input_text = question  + ' \n\n' + ' \n\n\n\n'.join(reasoning_steps[:i]) + ' \n\n\n\n' + step + ' \n\n\n\n'
        with torch.no_grad():
            prm_input = torch.tensor([prm_tokenizer.encode(prm_input_text)]).to(prm.device)
            prm_logits = prm(prm_input).logits[:,:,candidate_tokens]
            #print(logits.shape)
            prm_scores = prm_logits.softmax(dim=-1)[:,:,1]
            #print(scores.shape)
            step_scores = prm_scores[prm_input == 23535]
            step_probs  = step_scores.tolist()
        #print(question)
        print(f"Step: {step} \nScore: {step_probs}")

In [175]:
### incorrect case
question = "Bryan starts exercising at home during quarantine. To start, he decides to do 3 sets of 15 push-ups each. Near the end of the third set, he gets tired and does 5 fewer push-ups. How many push-ups did he do in total? Explain your reasoning step-by-step. Your final answer should be put between two ##, like ## 1 ## (if your final answer is 1), at the end of your response."

reasoning = "Let's break it down step-by-step!\n\n1. Bryan starts with 3 sets of 15 push-ups each. So, he does 3 x 15 = 45 push-ups in the first two sets.\n2. In the third set, he does 5 fewer push-ups than usual. So, he does 15 - 5 = 10 push-ups in the third set.\n3. To find the total number of push-ups, we add the number of push-ups in the first two sets (45) to the number of push-ups in the third set (10).\n\n45 + 10 = 55\n\n## 55 ##"

### basic
basic_prm_score(question, reasoning, prm, prm_tokenizer)

### prompt_cur_step
prompt_cur_step_prm_score(question, reasoning, prm, prm_tokenizer)

### prompt_all_step
prompt_all_step_prm_score(question, reasoning, prm, prm_tokenizer)

--------------------------------------------------
# steps: 6
Step scores: tensor([0.9492, 0.6094, 0.9648, 0.9727, 0.9414, 0.9414], device='cuda:0',
       dtype=torch.bfloat16)
--------------------------------------------------
# steps: 6
Step: Let's break it down step-by-step! 
Score: [0.859375]
Step: 1. Bryan starts with 3 sets of 15 push-ups each. So, he does 3 x 15 = 45 push-ups in the first two sets. 
Score: [0.94921875, 0.4921875]
Step: 2. In the third set, he does 5 fewer push-ups than usual. So, he does 15 - 5 = 10 push-ups in the third set. 
Score: [0.94921875, 0.60546875, 0.765625]
Step: 3. To find the total number of push-ups, we add the number of push-ups in the first two sets (45) to the number of push-ups in the third set (10). 
Score: [0.94921875, 0.609375, 0.96484375, 0.6796875]
Step: 45 + 10 = 55 
Score: [0.94921875, 0.609375, 0.96484375, 0.97265625, 0.64453125]
Step: ## 55 ## 
Score: [0.94921875, 0.609375, 0.96484375, 0.97265625, 0.94140625, 0.58203125]
-------------

In [176]:
### incorrect case
question = "Dylan's mother is throwing a baby shower for her best friend. She is expecting 40 guests, of whom she has cleared the parking lot to park in to, leaving only her car and her husband's jeep in the parking lot. The 40 guests, though, arrive in only 10 cars that they park in the parking lot.  If each car has 4 wheels, how many car wheels are there in the parking lot, including both of Dylan's parent's car wheels? Explain your reasoning step-by-step. Your final answer should be put between two ##, like ## 1 ## (if your final answer is 1), at the end of your response."

reasoning = "Let's break this down step by step!\n\n1. There are 40 guests arriving in 10 cars. This means that each car has 40/10 = 4 guests.\n2. Each car has 4 wheels, so each car has a total of 4 x 4 = 16 wheels.\n3. There are 10 cars in total, so the total number of wheels is 10 x 16 = 160 wheels.\n4. Dylan's mother and father have 2 cars in the parking lot, which are their own car and their husband's jeep. Each of these cars has 4 wheels, so the total number of wheels for these 2 cars is 2 x 4 = 8 wheels.\n5. Adding the wheels from the 10 guest cars (160) to the wheels from Dylan's parents' cars (8), we get a total of 160 + 8 = 168 wheels in the parking lot.\n\n## 168 ##"

### basic
basic_prm_score(question, reasoning, prm, prm_tokenizer)

### prompt_cur_step
prompt_cur_step_prm_score(question, reasoning, prm, prm_tokenizer)

### prompt_all_step
prompt_all_step_prm_score(question, reasoning, prm, prm_tokenizer)

--------------------------------------------------
# steps: 7
Step scores: tensor([0.9219, 0.8789, 0.9102, 0.7930, 0.6523, 0.9336, 0.8945],
       device='cuda:0', dtype=torch.bfloat16)
--------------------------------------------------
# steps: 7
Step: Let's break this down step by step! 
Score: [0.80859375]
Step: 1. There are 40 guests arriving in 10 cars. This means that each car has 40/10 = 4 guests. 
Score: [0.91796875, 0.6796875]
Step: 2. Each car has 4 wheels, so each car has a total of 4 x 4 = 16 wheels. 
Score: [0.921875, 0.87890625, 0.61328125]
Step: 3. There are 10 cars in total, so the total number of wheels is 10 x 16 = 160 wheels. 
Score: [0.921875, 0.87890625, 0.90625, 0.388671875]
Step: 4. Dylan's mother and father have 2 cars in the parking lot, which are their own car and their husband's jeep. Each of these cars has 4 wheels, so the total number of wheels for these 2 cars is 2 x 4 = 8 wheels. 
Score: [0.921875, 0.87890625, 0.91015625, 0.79296875, 0.58984375]
Step: 5. 

In [177]:
### correct case
question = "Jane picked 64 apples at the orchard. She sorted them into 4 different baskets to send to her friends. When Jane wasn't looking her sister took 3 apples from each of the baskets. How many apples are in each basket now? Explain your reasoning step-by-step. Your final answer should be put between two ##, like ## 1 ## (if your final answer is 1), at the end of your response."

reasoning = "Let's break this problem down step by step!\n\n1. Jane picked 64 apples and sorted them into 4 baskets. To find out how many apples are in each basket initially, we can divide the total number of apples (64) by the number of baskets (4):\n\n64 \u00f7 4 = 16\n\nSo, each basket initially had 16 apples.\n\n2. Jane's sister took 3 apples from each of the baskets. To find out how many apples are left in each basket, we need to subtract 3 from the initial number of apples in each basket (16):\n\n16 - 3 = 13\n\nSo, each basket now has 13 apples.\n\n## 13 ##"

### basic
basic_prm_score(question, reasoning, prm, prm_tokenizer)

### prompt_cur_step
prompt_cur_step_prm_score(question, reasoning, prm, prm_tokenizer)

### prompt_all_step
prompt_all_step_prm_score(question, reasoning, prm, prm_tokenizer)

--------------------------------------------------
# steps: 8
Step scores: tensor([0.9570, 0.9258, 0.9922, 0.9961, 0.9883, 0.9922, 0.9922, 0.9609],
       device='cuda:0', dtype=torch.bfloat16)
--------------------------------------------------
# steps: 8
Step: Let's break this problem down step by step! 
Score: [0.85546875]
Step: 1. Jane picked 64 apples and sorted them into 4 baskets. To find out how many apples are in each basket initially, we can divide the total number of apples (64) by the number of baskets (4): 
Score: [0.95703125, 0.7578125]
Step: 64 ÷ 4 = 16 
Score: [0.95703125, 0.92578125, 0.75390625]
Step: So, each basket initially had 16 apples. 
Score: [0.95703125, 0.921875, 0.9921875, 0.73828125]
Step: 2. Jane's sister took 3 apples from each of the baskets. To find out how many apples are left in each basket, we need to subtract 3 from the initial number of apples in each basket (16): 
Score: [0.95703125, 0.92578125, 0.9921875, 0.99609375, 0.89453125]
Step: 16 - 3 = 13 


In [178]:
### correct case
question = "Tim rides his bike back and forth to work for each of his 5 workdays.  His work is 20 miles away.  He also goes for a weekend bike ride of 200 miles.    If he can bike at 25 mph how much time does he spend biking a week? Explain your reasoning step-by-step. Your final answer should be put between two ##, like ## 1 ## (if your final answer is 1), at the end of your response."

reasoning = "Let's break this down step by step.\n\n1. Tim rides his bike to and from work for 5 days. This means he rides a total of 20 miles * 2 (there and back) * 5 days = 200 miles.\n2. He also goes for a weekend bike ride of 200 miles.\n3. To find the total distance Tim bikes in a week, we add the distance he bikes to and from work (200 miles) to the distance he bikes on the weekend (200 miles). This gives us a total distance of 200 + 200 = 400 miles.\n4. Tim bikes at a speed of 25 mph. To find the time it takes him to bike a certain distance, we can use the formula: time = distance / speed.\n5. Plugging in the values, we get: time = 400 miles / 25 mph = 16 hours.\n\n## 16 ##"

### basic
basic_prm_score(question, reasoning, prm, prm_tokenizer)

### prompt_cur_step
prompt_cur_step_prm_score(question, reasoning, prm, prm_tokenizer)

### prompt_all_step
prompt_all_step_prm_score(question, reasoning, prm, prm_tokenizer)

--------------------------------------------------
# steps: 7
Step scores: tensor([0.9609, 0.9375, 0.9961, 0.9297, 0.9688, 0.9766, 0.9297],
       device='cuda:0', dtype=torch.bfloat16)
--------------------------------------------------
# steps: 7
Step: Let's break this down step by step. 
Score: [0.87890625]
Step: 1. Tim rides his bike to and from work for 5 days. This means he rides a total of 20 miles * 2 (there and back) * 5 days = 200 miles. 
Score: [0.9609375, 0.8125]
Step: 2. He also goes for a weekend bike ride of 200 miles. 
Score: [0.9609375, 0.9375, 0.796875]
Step: 3. To find the total distance Tim bikes in a week, we add the distance he bikes to and from work (200 miles) to the distance he bikes on the weekend (200 miles). This gives us a total distance of 200 + 200 = 400 miles. 
Score: [0.9609375, 0.9375, 0.99609375, 0.72265625]
Step: 4. Tim bikes at a speed of 25 mph. To find the time it takes him to bike a certain distance, we can use the formula: time = distance / speed

In [179]:
### correct case
question = "Noah is a painter. He paints pictures and sells them at the park. He charges $60 for a large painting and $30 for a small painting. Last month he sold eight large paintings and four small paintings. If he sold twice as much this month, how much is his sales for this month? Explain your reasoning step-by-step. Your final answer should be put between two ##, like ## 1 ## (if your final answer is 1), at the end of your response."
reasoning = "Let's break it down step by step!\n\nLast month, Noah sold:\n\n* 8 large paintings at $60 each = 8 x $60 = $480\n* 4 small paintings at $30 each = 4 x $30 = $120\n\nTotal sales last month = $480 + $120 = $600\n\nThis month, Noah sold twice as much as last month. So, he sold:\n\n* 2 x 8 = 16 large paintings at $60 each = 16 x $60 = $960\n* 2 x 4 = 8 small paintings at $30 each = 8 x $30 = $240\n\nTotal sales this month = $960 + $240 = ## $1,200 ##"

### basic
basic_prm_score(question, reasoning, prm, prm_tokenizer)

### prompt_cur_step
prompt_cur_step_prm_score(question, reasoning, prm, prm_tokenizer)

### prompt_all_step
prompt_all_step_prm_score(question, reasoning, prm, prm_tokenizer)

--------------------------------------------------
# steps: 9
Step scores: tensor([0.9531, 0.6016, 0.9688, 0.9961, 0.9922, 0.9023, 0.9766, 0.9844, 0.9102],
       device='cuda:0', dtype=torch.bfloat16)
--------------------------------------------------
# steps: 9
Step: Let's break it down step by step! 
Score: [0.87890625]
Step: Last month, Noah sold: 
Score: [0.953125, 0.45703125]
Step: * 8 large paintings at $60 each = 8 x $60 = $480 
Score: [0.953125, 0.59375, 0.7890625]
Step: * 4 small paintings at $30 each = 4 x $30 = $120 
Score: [0.953125, 0.59375, 0.96875, 0.87890625]
Step: Total sales last month = $480 + $120 = $600 
Score: [0.953125, 0.5859375, 0.96875, 0.99609375, 0.7890625]
Step: This month, Noah sold twice as much as last month. So, he sold: 
Score: [0.953125, 0.5859375, 0.96875, 0.99609375, 0.9921875, 0.671875]
Step: * 2 x 8 = 16 large paintings at $60 each = 16 x $60 = $960 
Score: [0.953125, 0.59375, 0.96875, 0.99609375, 0.9921875, 0.8984375, 0.6796875]
Step: * 2 x 4 = 

In [121]:
### correct case
question = "Emery and Serena go to their school library, and each borrows a copy of The life of Jack Steve's book to read for their school project. If Emery can read five times as fast as Serena, and the book takes her 20 days to read, what's the average number of days the two take to read the book? Explain your reasoning step-by-step. Your final answer should be put between two ##, like ## 1 ## (if your final answer is 1), at the end of your response."

reasoning = "Let's break this problem down step by step!\n\n1. Emery can read five times as fast as Serena, which means that if Serena takes x days to read the book, Emery will take x/5 days to read the same book.\n2. Since Emery takes 20 days to read the book, we can set up the equation:\n\nx/5 = 20\n\nTo solve for x, we can multiply both sides by 5:\n\nx = 20 \u00d7 5\nx = 100\n\nSo Serena takes 100 days to read the book.\n\n3. To find the average number of days it takes for both Emery and Serena to read the book, we need to add their reading times and divide by 2:\n\nAverage reading time = (20 + 100) / 2\nAverage reading time = 120 / 2\nAverage reading time = 60\n\n## 60 ##"

### basic
basic_prm_score(question, reasoning, prm, prm_tokenizer)

### prompt_cur_step
prompt_cur_step_prm_score(question, reasoning, prm, prm_tokenizer)

### prompt_all_step
prompt_all_step_prm_score(question, reasoning, prm, prm_tokenizer)

--------------------------------------------------
# steps: 13
Step scores: tensor([0.9531, 0.8906, 0.7461, 0.8281, 0.9688, 0.9844, 0.9922, 0.9922, 0.9648,
        0.9805, 0.9062, 0.9141, 0.9102], device='cuda:0', dtype=torch.bfloat16)
--------------------------------------------------
# steps: 13
Step: Let's break this problem down step by step! 
Score: [0.81640625]
Step: 1. Emery can read five times as fast as Serena, which means that if Serena takes x days to read the book, Emery will take x/5 days to read the same book. 
Score: [0.953125, 0.6875]
Step: 2. Since Emery takes 20 days to read the book, we can set up the equation: 
Score: [0.94921875, 0.89453125, 0.69140625]
Step: x/5 = 20 
Score: [0.94921875, 0.89453125, 0.75390625, 0.6796875]
Step: To solve for x, we can multiply both sides by 5: 
Score: [0.94921875, 0.89453125, 0.75390625, 0.828125, 0.8359375]
Step: x = 20 × 5 
Score: [0.953125, 0.890625, 0.74609375, 0.83203125, 0.96875, 0.80078125]
Step: x = 100 
Score: [0.953125, 0

## Re-evaluate without steps under the threshold

In [161]:
def remove_prompt_cur_step_prm_score(question, reasoning, prm, prm_tokenizer, threshold):
    print("-"*50)
    
    reasoning_steps = [l.strip() for l in reasoning.split("\n") if l.strip()]
    print(f"# steps: {len(reasoning_steps)}")
    
    ### first probabilities
    first_step_probs = []
    for i, step in enumerate(reasoning_steps):
        if i == 0:
            prm_input_text = question + ' \n\n' + f"Based on problem, next step is: {step}" + ' \n\n\n\n'
        else:
            prm_input_text = question  + ' \n\n' + ' \n\n\n\n'.join(reasoning_steps[:i]) + ' \n\n\n\n' + f" Based on problem and previous reasoning, next step is: {step}" + ' \n\n\n\n'
        with torch.no_grad():
            prm_input = torch.tensor([prm_tokenizer.encode(prm_input_text)]).to(prm.device)
            prm_logits = prm(prm_input).logits[:,:,candidate_tokens]
            #print(logits.shape)
            prm_scores = prm_logits.softmax(dim=-1)[:,:,1]
            #print(scores.shape)
            step_scores = prm_scores[prm_input == 23535]
            step_probs  = step_scores.tolist()
        first_step_probs.append(step_probs[-1])
        #print(question)
        print(f"Step: {step} \nScore: {step_probs}")
    
    ### remove steps under threshold
    under_threshold = [i for i, p in enumerate(first_step_probs) if p < 0.6]
    print(f"under threshold index: {under_threshold}")
    if under_threshold:
        reasoning_steps = [s for i, s in enumerate(reasoning_steps) if i not in under_threshold]
        print(f"New reasoning steps: {reasoning_steps}")
        
        ### second probabilities
        print("-"*50)
        print(f"# steps: {len(reasoning_steps)}")
        
        for i, step in enumerate(reasoning_steps):
            if i == 0:
                prm_input_text = question + ' \n\n' + "Based on problem, next step is: " + step + ' \n\n\n\n'
            else:
                prm_input_text = question  + ' \n\n' + ' \n\n\n\n'.join(reasoning_steps[:i]) + ' \n\n\n\n' + " Based on problem and previous reasoning, next step is: " + step + ' \n\n\n\n'
            with torch.no_grad():
                prm_input = torch.tensor([prm_tokenizer.encode(prm_input_text)]).to(prm.device)
                prm_logits = prm(prm_input).logits[:,:,candidate_tokens]
                #print(logits.shape)
                prm_scores = prm_logits.softmax(dim=-1)[:,:,1]
                #print(scores.shape)
                step_scores = prm_scores[prm_input == 23535]
                step_probs  = step_scores.tolist()
            #print(question)
            print(f"Step: {step} \nScore: {step_probs}")

In [162]:
def remove_prompt_all_step_prm_score(question, reasoning, prm, prm_tokenizer, threshold):
    print("-"*50)
    
    reasoning_steps = [l.strip() for l in reasoning.split("\n") if l.strip()]
    prefix = ["Based on problem, next step is: ", " Based on problem and previous reasoning, next step is: "]
    reasoning_steps = [prefix[min(i,1)] + f"{step}" for i, step in enumerate(reasoning_steps)]
    print(f"# steps: {len(reasoning_steps)}")
    
    ### first probabilities
    first_step_probs = []
    for i, step in enumerate(reasoning_steps):
        if i == 0:
            prm_input_text = question + ' \n\n' + f"{step}" + ' \n\n\n\n'
        else:
            prm_input_text = question  + ' \n\n' + ' \n\n\n\n'.join(reasoning_steps[:i]) + ' \n\n\n\n' + f"{step}" + ' \n\n\n\n'
        with torch.no_grad():
            prm_input = torch.tensor([prm_tokenizer.encode(prm_input_text)]).to(prm.device)
            prm_logits = prm(prm_input).logits[:,:,candidate_tokens]
            #print(logits.shape)
            prm_scores = prm_logits.softmax(dim=-1)[:,:,1]
            #print(scores.shape)
            step_scores = prm_scores[prm_input == 23535]
            step_probs  = step_scores.tolist()
        first_step_probs.append(step_probs[-1])
        #print(question)
        print(f"Step: {step} \nScore: {step_probs}")
    
    ### remove steps under threshold
    under_threshold = [i for i, p in enumerate(first_step_probs) if p < 0.6]
    print(f"under threshold index: {under_threshold}")
    if under_threshold:
        reasoning_steps = [s for i, s in enumerate(reasoning_steps) if i not in under_threshold]
        print(f"New reasoning steps: {reasoning_steps}")
        
        ### second probabilities
        print("-"*50)
        print(f"# steps: {len(reasoning_steps)}")
        
        for i, step in enumerate(reasoning_steps):
            if i == 0:
                prm_input_text = question + ' \n\n' + f"{step}" + ' \n\n\n\n'
            else:
                prm_input_text = question  + ' \n\n' + ' \n\n\n\n'.join(reasoning_steps[:i]) + ' \n\n\n\n' + f"{step}" + ' \n\n\n\n'
            with torch.no_grad():
                prm_input = torch.tensor([prm_tokenizer.encode(prm_input_text)]).to(prm.device)
                prm_logits = prm(prm_input).logits[:,:,candidate_tokens]
                #print(logits.shape)
                prm_scores = prm_logits.softmax(dim=-1)[:,:,1]
                #print(scores.shape)
                step_scores = prm_scores[prm_input == 23535]
                step_probs  = step_scores.tolist()
            #print(question)
            print(f"Step: {step} \nScore: {step_probs}")

In [164]:
### incorrect case
question = "Bryan starts exercising at home during quarantine. To start, he decides to do 3 sets of 15 push-ups each. Near the end of the third set, he gets tired and does 5 fewer push-ups. How many push-ups did he do in total? Explain your reasoning step-by-step. Your final answer should be put between two ##, like ## 1 ## (if your final answer is 1), at the end of your response."

reasoning = "Let's break it down step-by-step!\n\n1. Bryan starts with 3 sets of 15 push-ups each. So, he does 3 x 15 = 45 push-ups in the first two sets.\n2. In the third set, he does 5 fewer push-ups than usual. So, he does 15 - 5 = 10 push-ups in the third set.\n3. To find the total number of push-ups, we add the number of push-ups in the first two sets (45) to the number of push-ups in the third set (10).\n\n45 + 10 = 55\n\n## 55 ##"

### basic
basic_prm_score(question, reasoning, prm, prm_tokenizer)

### prompt_cur_step & remove_prompt_cur_step
remove_prompt_cur_step_prm_score(question, reasoning, prm, prm_tokenizer, threshold=0.6)

### prompt_all_step & remove_prompt_all_step
#remove_prompt_all_step_prm_score(question, reasoning, prm, prm_tokenizer, threshold=0.6)


--------------------------------------------------
# steps: 6
Step scores: tensor([0.9492, 0.6094, 0.9648, 0.9727, 0.9414, 0.9414], device='cuda:0',
       dtype=torch.bfloat16)
--------------------------------------------------
# steps: 6
Step: Let's break it down step-by-step! 
Score: [0.80859375]
Step: 1. Bryan starts with 3 sets of 15 push-ups each. So, he does 3 x 15 = 45 push-ups in the first two sets. 
Score: [0.94921875, 0.49609375]
Step: 2. In the third set, he does 5 fewer push-ups than usual. So, he does 15 - 5 = 10 push-ups in the third set. 
Score: [0.94921875, 0.60546875, 0.76171875]
Step: 3. To find the total number of push-ups, we add the number of push-ups in the first two sets (45) to the number of push-ups in the third set (10). 
Score: [0.94921875, 0.609375, 0.96484375, 0.6796875]
Step: 45 + 10 = 55 
Score: [0.94921875, 0.609375, 0.96484375, 0.97265625, 0.79296875]
Step: ## 55 ## 
Score: [0.94921875, 0.609375, 0.96484375, 0.97265625, 0.94140625, 0.70703125]
under th

In [156]:
### incorrect case
question = "Dylan's mother is throwing a baby shower for her best friend. She is expecting 40 guests, of whom she has cleared the parking lot to park in to, leaving only her car and her husband's jeep in the parking lot. The 40 guests, though, arrive in only 10 cars that they park in the parking lot.  If each car has 4 wheels, how many car wheels are there in the parking lot, including both of Dylan's parent's car wheels? Explain your reasoning step-by-step. Your final answer should be put between two ##, like ## 1 ## (if your final answer is 1), at the end of your response."

reasoning = "Let's break this down step by step!\n\n1. There are 40 guests arriving in 10 cars. This means that each car has 40/10 = 4 guests.\n2. Each car has 4 wheels, so each car has a total of 4 x 4 = 16 wheels.\n3. There are 10 cars in total, so the total number of wheels is 10 x 16 = 160 wheels.\n4. Dylan's mother and father have 2 cars in the parking lot, which are their own car and their husband's jeep. Each of these cars has 4 wheels, so the total number of wheels for these 2 cars is 2 x 4 = 8 wheels.\n5. Adding the wheels from the 10 guest cars (160) to the wheels from Dylan's parents' cars (8), we get a total of 160 + 8 = 168 wheels in the parking lot.\n\n## 168 ##"

### basic
basic_prm_score(question, reasoning, prm, prm_tokenizer)

### prompt_cur_step & remove_prompt_cur_step
remove_prompt_cur_step_prm_score(question, reasoning, prm, prm_tokenizer, threshold=0.6)

### prompt_all_step & remove_prompt_all_step
#remove_prompt_all_step_prm_score(question, reasoning, prm, prm_tokenizer, threshold=0.6)

--------------------------------------------------
# steps: 7
Step scores: tensor([0.9219, 0.8789, 0.9102, 0.7930, 0.6523, 0.9336, 0.8945],
       device='cuda:0', dtype=torch.bfloat16)
--------------------------------------------------
# steps: 7
Step: Let's break this down step by step! 
Score: [0.79296875]
Step: 1. There are 40 guests arriving in 10 cars. This means that each car has 40/10 = 4 guests. 
Score: [0.91796875, 0.609375]
Step: 2. Each car has 4 wheels, so each car has a total of 4 x 4 = 16 wheels. 
Score: [0.91796875, 0.875, 0.69921875]
Step: 3. There are 10 cars in total, so the total number of wheels is 10 x 16 = 160 wheels. 
Score: [0.921875, 0.87890625, 0.90625, 0.27734375]
Step: 4. Dylan's mother and father have 2 cars in the parking lot, which are their own car and their husband's jeep. Each of these cars has 4 wheels, so the total number of wheels for these 2 cars is 2 x 4 = 8 wheels. 
Score: [0.921875, 0.87890625, 0.90625, 0.78515625, 0.5]
Step: 5. Adding the whee

In [157]:
### correct case
question = "Jane picked 64 apples at the orchard. She sorted them into 4 different baskets to send to her friends. When Jane wasn't looking her sister took 3 apples from each of the baskets. How many apples are in each basket now? Explain your reasoning step-by-step. Your final answer should be put between two ##, like ## 1 ## (if your final answer is 1), at the end of your response."

reasoning = "Let's break this problem down step by step!\n\n1. Jane picked 64 apples and sorted them into 4 baskets. To find out how many apples are in each basket initially, we can divide the total number of apples (64) by the number of baskets (4):\n\n64 \u00f7 4 = 16\n\nSo, each basket initially had 16 apples.\n\n2. Jane's sister took 3 apples from each of the baskets. To find out how many apples are left in each basket, we need to subtract 3 from the initial number of apples in each basket (16):\n\n16 - 3 = 13\n\nSo, each basket now has 13 apples.\n\n## 13 ##"

### basic
basic_prm_score(question, reasoning, prm, prm_tokenizer)

### prompt_cur_step & remove_prompt_cur_step
remove_prompt_cur_step_prm_score(question, reasoning, prm, prm_tokenizer, threshold=0.6)

### prompt_all_step & remove_prompt_all_step
#remove_prompt_all_step_prm_score(question, reasoning, prm, prm_tokenizer, threshold=0.6)

--------------------------------------------------
# steps: 8
Step scores: tensor([0.9570, 0.9258, 0.9922, 0.9961, 0.9883, 0.9922, 0.9922, 0.9609],
       device='cuda:0', dtype=torch.bfloat16)
--------------------------------------------------
# steps: 8
Step: Let's break this problem down step by step! 
Score: [0.8203125]
Step: 1. Jane picked 64 apples and sorted them into 4 baskets. To find out how many apples are in each basket initially, we can divide the total number of apples (64) by the number of baskets (4): 
Score: [0.95703125, 0.7578125]
Step: 64 ÷ 4 = 16 
Score: [0.95703125, 0.92578125, 0.8515625]
Step: So, each basket initially had 16 apples. 
Score: [0.95703125, 0.92578125, 0.9921875, 0.8515625]
Step: 2. Jane's sister took 3 apples from each of the baskets. To find out how many apples are left in each basket, we need to subtract 3 from the initial number of apples in each basket (16): 
Score: [0.95703125, 0.92578125, 0.9921875, 0.99609375, 0.91796875]
Step: 16 - 3 = 13 
S

In [158]:
### correct case
question = "Tim rides his bike back and forth to work for each of his 5 workdays.  His work is 20 miles away.  He also goes for a weekend bike ride of 200 miles.    If he can bike at 25 mph how much time does he spend biking a week? Explain your reasoning step-by-step. Your final answer should be put between two ##, like ## 1 ## (if your final answer is 1), at the end of your response."

reasoning = "Let's break this down step by step.\n\n1. Tim rides his bike to and from work for 5 days. This means he rides a total of 20 miles * 2 (there and back) * 5 days = 200 miles.\n2. He also goes for a weekend bike ride of 200 miles.\n3. To find the total distance Tim bikes in a week, we add the distance he bikes to and from work (200 miles) to the distance he bikes on the weekend (200 miles). This gives us a total distance of 200 + 200 = 400 miles.\n4. Tim bikes at a speed of 25 mph. To find the time it takes him to bike a certain distance, we can use the formula: time = distance / speed.\n5. Plugging in the values, we get: time = 400 miles / 25 mph = 16 hours.\n\n## 16 ##"

### basic
basic_prm_score(question, reasoning, prm, prm_tokenizer)

### prompt_cur_step & remove_prompt_cur_step
remove_prompt_cur_step_prm_score(question, reasoning, prm, prm_tokenizer, threshold=0.6)

### prompt_all_step & remove_prompt_all_step
#remove_prompt_all_step_prm_score(question, reasoning, prm, prm_tokenizer, threshold=0.6)

--------------------------------------------------
# steps: 7
Step scores: tensor([0.9609, 0.9375, 0.9961, 0.9297, 0.9688, 0.9766, 0.9297],
       device='cuda:0', dtype=torch.bfloat16)
--------------------------------------------------
# steps: 7
Step: Let's break this down step by step. 
Score: [0.8515625]
Step: 1. Tim rides his bike to and from work for 5 days. This means he rides a total of 20 miles * 2 (there and back) * 5 days = 200 miles. 
Score: [0.9609375, 0.7734375]
Step: 2. He also goes for a weekend bike ride of 200 miles. 
Score: [0.9609375, 0.9375, 0.828125]
Step: 3. To find the total distance Tim bikes in a week, we add the distance he bikes to and from work (200 miles) to the distance he bikes on the weekend (200 miles). This gives us a total distance of 200 + 200 = 400 miles. 
Score: [0.9609375, 0.9375, 0.99609375, 0.7578125]
Step: 4. Tim bikes at a speed of 25 mph. To find the time it takes him to bike a certain distance, we can use the formula: time = distance / spee

In [159]:
### correct case
question = "Noah is a painter. He paints pictures and sells them at the park. He charges $60 for a large painting and $30 for a small painting. Last month he sold eight large paintings and four small paintings. If he sold twice as much this month, how much is his sales for this month? Explain your reasoning step-by-step. Your final answer should be put between two ##, like ## 1 ## (if your final answer is 1), at the end of your response."
reasoning = "Let's break it down step by step!\n\nLast month, Noah sold:\n\n* 8 large paintings at $60 each = 8 x $60 = $480\n* 4 small paintings at $30 each = 4 x $30 = $120\n\nTotal sales last month = $480 + $120 = $600\n\nThis month, Noah sold twice as much as last month. So, he sold:\n\n* 2 x 8 = 16 large paintings at $60 each = 16 x $60 = $960\n* 2 x 4 = 8 small paintings at $30 each = 8 x $30 = $240\n\nTotal sales this month = $960 + $240 = ## $1,200 ##"

### basic
basic_prm_score(question, reasoning, prm, prm_tokenizer)

### prompt_cur_step & remove_prompt_cur_step
remove_prompt_cur_step_prm_score(question, reasoning, prm, prm_tokenizer, threshold=0.6)

### prompt_all_step & remove_prompt_all_step
#remove_prompt_all_step_prm_score(question, reasoning, prm, prm_tokenizer, threshold=0.6)

--------------------------------------------------
# steps: 9
Step scores: tensor([0.9531, 0.6016, 0.9688, 0.9961, 0.9922, 0.9023, 0.9766, 0.9844, 0.9102],
       device='cuda:0', dtype=torch.bfloat16)
--------------------------------------------------
# steps: 9
Step: Let's break it down step by step! 
Score: [0.8046875]
Step: Last month, Noah sold: 
Score: [0.953125, 0.57421875]
Step: * 8 large paintings at $60 each = 8 x $60 = $480 
Score: [0.953125, 0.59375, 0.80078125]
Step: * 4 small paintings at $30 each = 4 x $30 = $120 
Score: [0.953125, 0.59375, 0.96875, 0.89453125]
Step: Total sales last month = $480 + $120 = $600 
Score: [0.953125, 0.59375, 0.96875, 0.99609375, 0.88671875]
Step: This month, Noah sold twice as much as last month. So, he sold: 
Score: [0.953125, 0.5859375, 0.96875, 0.99609375, 0.9921875, 0.69921875]
Step: * 2 x 8 = 16 large paintings at $60 each = 16 x $60 = $960 
Score: [0.953125, 0.59375, 0.96875, 0.99609375, 0.9921875, 0.8984375, 0.62890625]
Step: * 2 x 4 

In [160]:
### correct case
question = "Emery and Serena go to their school library, and each borrows a copy of The life of Jack Steve's book to read for their school project. If Emery can read five times as fast as Serena, and the book takes her 20 days to read, what's the average number of days the two take to read the book? Explain your reasoning step-by-step. Your final answer should be put between two ##, like ## 1 ## (if your final answer is 1), at the end of your response."

reasoning = "Let's break this problem down step by step!\n\n1. Emery can read five times as fast as Serena, which means that if Serena takes x days to read the book, Emery will take x/5 days to read the same book.\n2. Since Emery takes 20 days to read the book, we can set up the equation:\n\nx/5 = 20\n\nTo solve for x, we can multiply both sides by 5:\n\nx = 20 \u00d7 5\nx = 100\n\nSo Serena takes 100 days to read the book.\n\n3. To find the average number of days it takes for both Emery and Serena to read the book, we need to add their reading times and divide by 2:\n\nAverage reading time = (20 + 100) / 2\nAverage reading time = 120 / 2\nAverage reading time = 60\n\n## 60 ##"

### basic
basic_prm_score(question, reasoning, prm, prm_tokenizer)

### prompt_cur_step & remove_prompt_cur_step
remove_prompt_cur_step_prm_score(question, reasoning, prm, prm_tokenizer, threshold=0.6)

### prompt_all_step & remove_prompt_all_step
#remove_prompt_all_step_prm_score(question, reasoning, prm, prm_tokenizer, threshold=0.6)

--------------------------------------------------
# steps: 13
Step scores: tensor([0.9531, 0.8906, 0.7461, 0.8281, 0.9688, 0.9844, 0.9922, 0.9922, 0.9648,
        0.9805, 0.9062, 0.9141, 0.9102], device='cuda:0', dtype=torch.bfloat16)
--------------------------------------------------
# steps: 13
Step: Let's break this problem down step by step! 
Score: [0.81640625]
Step: 1. Emery can read five times as fast as Serena, which means that if Serena takes x days to read the book, Emery will take x/5 days to read the same book. 
Score: [0.953125, 0.6875]
Step: 2. Since Emery takes 20 days to read the book, we can set up the equation: 
Score: [0.94921875, 0.89453125, 0.69140625]
Step: x/5 = 20 
Score: [0.94921875, 0.89453125, 0.75390625, 0.6796875]
Step: To solve for x, we can multiply both sides by 5: 
Score: [0.94921875, 0.89453125, 0.75390625, 0.828125, 0.8359375]
Step: x = 20 × 5 
Score: [0.953125, 0.890625, 0.74609375, 0.83203125, 0.96875, 0.80078125]
Step: x = 100 
Score: [0.953125, 0

## Prompt to judge the correctness of current step

In [9]:
def prompt_judge_cur_step_prm_score(question, reasoning, prm, prm_tokenizer):
    print("-"*50)
    
    reasoning_steps = [l.strip() for l in reasoning.split("\n") if l.strip()]
    print(f"# steps: {len(reasoning_steps)}")
    
    for i, step in enumerate(reasoning_steps):
        if i == 0:
            prm_input_text = question + ' \n\n' + f"Based on problem, next step \"{step}\" is correct." + ' \n\n\n\n'
        else:
            prm_input_text = question  + ' \n\n' + ' \n\n\n\n'.join(reasoning_steps[:i]) + ' \n\n\n\n' + f" Based on problem and previous reasoning, next step \"{step}\" is correct." + ' \n\n\n\n'
            
        with torch.no_grad():
            prm_input = torch.tensor([prm_tokenizer.encode(prm_input_text)]).to(prm.device)
            prm_logits = prm(prm_input).logits[:,:,candidate_tokens]
            #print(logits.shape)
            prm_scores = prm_logits.softmax(dim=-1)[:,:,1]
            #print(scores.shape)
            step_scores = prm_scores[prm_input == 23535]
            step_probs  = step_scores.tolist()
        #print(question)
        print(f"Step: {step} \nScore: {step_probs}")

In [10]:
### incorrect case
question = "Bryan starts exercising at home during quarantine. To start, he decides to do 3 sets of 15 push-ups each. Near the end of the third set, he gets tired and does 5 fewer push-ups. How many push-ups did he do in total? Explain your reasoning step-by-step. Your final answer should be put between two ##, like ## 1 ## (if your final answer is 1), at the end of your response."

reasoning = "Let's break it down step-by-step!\n\n1. Bryan starts with 3 sets of 15 push-ups each. So, he does 3 x 15 = 45 push-ups in the first two sets.\n2. In the third set, he does 5 fewer push-ups than usual. So, he does 15 - 5 = 10 push-ups in the third set.\n3. To find the total number of push-ups, we add the number of push-ups in the first two sets (45) to the number of push-ups in the third set (10).\n\n45 + 10 = 55\n\n## 55 ##"

### basic
basic_prm_score(question, reasoning, prm, prm_tokenizer)

### prompt_judge_cur_step
prompt_judge_cur_step_prm_score(question, reasoning, prm, prm_tokenizer)

--------------------------------------------------
# steps: 6
Step scores: tensor([0.9492, 0.6094, 0.9648, 0.9727, 0.9414, 0.9414], device='cuda:0',
       dtype=torch.bfloat16)
--------------------------------------------------
# steps: 6
Step: Let's break it down step-by-step! 
Score: [0.7265625]
Step: 1. Bryan starts with 3 sets of 15 push-ups each. So, he does 3 x 15 = 45 push-ups in the first two sets. 
Score: [0.94921875, 0.3125]
Step: 2. In the third set, he does 5 fewer push-ups than usual. So, he does 15 - 5 = 10 push-ups in the third set. 
Score: [0.94921875, 0.60546875, 0.6484375]
Step: 3. To find the total number of push-ups, we add the number of push-ups in the first two sets (45) to the number of push-ups in the third set (10). 
Score: [0.94921875, 0.609375, 0.96484375, 0.61328125]
Step: 45 + 10 = 55 
Score: [0.94921875, 0.609375, 0.96484375, 0.97265625, 0.68359375]
Step: ## 55 ## 
Score: [0.94921875, 0.609375, 0.96484375, 0.97265625, 0.94140625, 0.64453125]


In [11]:
### incorrect case
question = "Dylan's mother is throwing a baby shower for her best friend. She is expecting 40 guests, of whom she has cleared the parking lot to park in to, leaving only her car and her husband's jeep in the parking lot. The 40 guests, though, arrive in only 10 cars that they park in the parking lot.  If each car has 4 wheels, how many car wheels are there in the parking lot, including both of Dylan's parent's car wheels? Explain your reasoning step-by-step. Your final answer should be put between two ##, like ## 1 ## (if your final answer is 1), at the end of your response."

reasoning = "Let's break this down step by step!\n\n1. There are 40 guests arriving in 10 cars. This means that each car has 40/10 = 4 guests.\n2. Each car has 4 wheels, so each car has a total of 4 x 4 = 16 wheels.\n3. There are 10 cars in total, so the total number of wheels is 10 x 16 = 160 wheels.\n4. Dylan's mother and father have 2 cars in the parking lot, which are their own car and their husband's jeep. Each of these cars has 4 wheels, so the total number of wheels for these 2 cars is 2 x 4 = 8 wheels.\n5. Adding the wheels from the 10 guest cars (160) to the wheels from Dylan's parents' cars (8), we get a total of 160 + 8 = 168 wheels in the parking lot.\n\n## 168 ##"

### basic
basic_prm_score(question, reasoning, prm, prm_tokenizer)

### prompt_judge_cur_step
prompt_judge_cur_step_prm_score(question, reasoning, prm, prm_tokenizer)

--------------------------------------------------
# steps: 7
Step scores: tensor([0.9219, 0.8789, 0.9102, 0.7930, 0.6523, 0.9336, 0.8945],
       device='cuda:0', dtype=torch.bfloat16)
--------------------------------------------------
# steps: 7
Step: Let's break this down step by step! 
Score: [0.77734375]
Step: 1. There are 40 guests arriving in 10 cars. This means that each car has 40/10 = 4 guests. 
Score: [0.91796875, 0.5859375]
Step: 2. Each car has 4 wheels, so each car has a total of 4 x 4 = 16 wheels. 
Score: [0.91796875, 0.875, 0.5703125]
Step: 3. There are 10 cars in total, so the total number of wheels is 10 x 16 = 160 wheels. 
Score: [0.921875, 0.87890625, 0.90625, 0.318359375]
Step: 4. Dylan's mother and father have 2 cars in the parking lot, which are their own car and their husband's jeep. Each of these cars has 4 wheels, so the total number of wheels for these 2 cars is 2 x 4 = 8 wheels. 
Score: [0.921875, 0.87890625, 0.90625, 0.79296875, 0.4609375]
Step: 5. Adding t

In [13]:
### correct case
question = "Jane picked 64 apples at the orchard. She sorted them into 4 different baskets to send to her friends. When Jane wasn't looking her sister took 3 apples from each of the baskets. How many apples are in each basket now? Explain your reasoning step-by-step. Your final answer should be put between two ##, like ## 1 ## (if your final answer is 1), at the end of your response."

reasoning = "Let's break this problem down step by step!\n\n1. Jane picked 64 apples and sorted them into 4 baskets. To find out how many apples are in each basket initially, we can divide the total number of apples (64) by the number of baskets (4):\n\n64 \u00f7 4 = 16\n\nSo, each basket initially had 16 apples.\n\n2. Jane's sister took 3 apples from each of the baskets. To find out how many apples are left in each basket, we need to subtract 3 from the initial number of apples in each basket (16):\n\n16 - 3 = 13\n\nSo, each basket now has 13 apples.\n\n## 13 ##"

### basic
basic_prm_score(question, reasoning, prm, prm_tokenizer)

### prompt_judge_cur_step
prompt_judge_cur_step_prm_score(question, reasoning, prm, prm_tokenizer)

--------------------------------------------------
# steps: 8
Step scores: tensor([0.9570, 0.9258, 0.9922, 0.9961, 0.9883, 0.9922, 0.9922, 0.9609],
       device='cuda:0', dtype=torch.bfloat16)
--------------------------------------------------
# steps: 8
Step: Let's break this problem down step by step! 
Score: [0.77734375]
Step: 1. Jane picked 64 apples and sorted them into 4 baskets. To find out how many apples are in each basket initially, we can divide the total number of apples (64) by the number of baskets (4): 
Score: [0.95703125, 0.5703125]
Step: 64 ÷ 4 = 16 
Score: [0.95703125, 0.92578125, 0.796875]
Step: So, each basket initially had 16 apples. 
Score: [0.95703125, 0.92578125, 0.9921875, 0.77734375]
Step: 2. Jane's sister took 3 apples from each of the baskets. To find out how many apples are left in each basket, we need to subtract 3 from the initial number of apples in each basket (16): 
Score: [0.95703125, 0.92578125, 0.9921875, 0.99609375, 0.6953125]
Step: 16 - 3 = 13 
S